# NLP Text Preprocessing - Urdu Tweets Dataset
**Students:** Raqib Hayat(NUM-BSCS-2022-40) and Abu Bakar(NUM-BSCS-2022-41)

**Course:** CSC-355 Natural Language Processing  

**Project:** Robust Sentiment and Emotion Classification on Noisy Urdu Tweets: A Multi-Stage NLP Pipeline on SentiUrdu-1M

**Dataset:** SentiUrdu-1M (Urdu Tweets Dataset)  

**University:** Namal University Mianwali  

## Install Required Libraries

In [1]:
!pip install emoji openpyxl pandas

   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/608.4 kB ? eta -:--:--
   ---------------------------------------- 608.4/608.4 kB 1.6 MB/s  0:00:00



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\abuba\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


## Import Libraries

In [2]:
import re
import unicodedata
import emoji
import pandas as pd

## Load the Dataset

Download dataset from here: https://data.mendeley.com/datasets/rz3xg97rm5/1

In [3]:
# Load the Urdu Tweets Dataset
df = pd.read_excel('Urdu Tweets Dataset.xlsx', engine='openpyxl')

print(f"Dataset Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nData Types:")
print(df.dtypes)
print(f"\nNull Values:")
print(df.isnull().sum())
print(f"\nFirst 5 rows:")
df.head()

Dataset Shape: (1048000, 4)
Columns: ['Id', 'Text', 'Emotions', 'Category']

Data Types:
Id           int64
Text        object
Emotions       str
Category       str
dtype: object

Null Values:
Id               0
Text             0
Emotions         0
Category    514571
dtype: int64

First 5 rows:


,Id,Text,Emotions,Category
0,1313165582020349952,.Assalam Alikum 🦋😊 اے ایمان والو میرے دشم...,"['SMILING FACE WITH SMILING EYES , 0.644']",Joy
1,1314653586509488128,- '' 🍀 🌾 🌴 '' - ﺍﮬﺪِﻧَﺎﻟﺼِّﺮَﺍﻁَ ﺍَﻟﻤُﺴﺘَﻘِﯿﻢ...,"['HEAVY BLACK HEART , 0.746', 'SPARKLES , 0.35...",NaN
2,1311404383071067904,💕 میں نہیں کرتی اُس کا ذکر کسی تیسرے کے سات...,"['TWO HEARTS , 0.632']",NaN
3,1313508198876433920,.🌿.. ﴿کے میشود تمامِ جـهان سـوےِ کربـلا با ک...,"['HERB , 0.384']",NaN
4,1313510191405596928,.🌿.. ﴿کے میشود تمامِ جـهان سـوےِ کربـلا با ک...,"['HERB , 0.384']",NaN


## Sample Data for Demonstration

Since the dataset has over 1 million tweets, we take a small sample for demonstrating each preprocessing step clearly.

In [4]:
# Take a sample of 5 tweets for step-by-step demonstration
sample_df = df[['Text']].dropna(subset=['Text']).head(5).copy()
sample_df = sample_df.reset_index(drop=True)

print("=" * 70)
print("SAMPLE RAW TWEETS")
print("=" * 70)
for i, row in sample_df.iterrows():
    print(f"\n[Tweet {i+1}]:")
    print(f"  {row['Text'][:200]}")

SAMPLE RAW TWEETS

[Tweet 1]:
       .Assalam Alikum 🦋😊 اے ایمان والو میرے دشمنوں اور اپنے دشمنوں کافروں کو دوست نہ بناؤ کہ ان کے پاس دوستی کے پ…

[Tweet 2]:
   - '' 🍀 🌾 🌴 '' - ﺍﮬﺪِﻧَﺎﻟﺼِّﺮَﺍﻁَ ﺍَﻟﻤُﺴﺘَﻘِﯿﻢَ '' ﺍﮮ ﻣﯿﺮﮮ ﺭﺏ، ﻣﺠﮭﮯ ﺳﯿﺪﮬﮯ ﺭﺍﺳﺘﮯ ﭘﺮ ﭼﻼ❤️السلام و عليكم.. صباح الخير✨ جمعہ مبارک..💝…

[Tweet 3]:
     💕 میں نہیں کرتی اُس کا ذکر کسی تیسرے کے ساتھ اُس کے بارے میں بات صرف خُدا سے ہوتی ہے   .💕 نـــور

[Tweet 4]:
    .🌿.. ﴿کے میشود تمامِ جـهان سـوےِ کربـلا با کاروانِ مهـدےِ موعد عج سفـر کنند﴾ حب_الحسین_یجمعنا

[Tweet 5]:
    .🌿.. ﴿کے میشود تمامِ جـهان سـوےِ کربـلا با کاروانِ مهـدےِ موعد عج سفـر کنند﴾  اربعین ما_ملت_امام_حسینیم حب_ال…


---

## Preprocessing Steps

The following preprocessing steps are applied in order. Each step is designed specifically for Urdu tweet data from the SentiUrdu-1M dataset.

### Step 1: Unicode Normalization (NFC)

**Why:** Urdu text uses Arabic script which can represent the same character using multiple Unicode code points (composed vs decomposed forms). Without normalization, identical words appear as different tokens, fragmenting vocabulary.

**What it does:** Applies NFC (Canonical Decomposition followed by Canonical Composition) to unify equivalent Unicode representations.

In [5]:
def normalize_unicode(text):
    """Apply NFC normalization to unify equivalent Unicode representations."""
    return unicodedata.normalize("NFC", text)


# Demonstrate on sample
print("STEP 1: Unicode Normalization")
print("=" * 70)
for i, row in sample_df.iterrows():
    original = row['Text'][:150]
    cleaned = normalize_unicode(original)
    print(f"\n[Tweet {i+1}]")
    print(f"  INPUT : {original}")
    print(f"  OUTPUT: {cleaned}")
    if original != cleaned:
        print(f"  >> Changed: Yes (Unicode codepoints unified)")
    else:
        print(f"  >> Changed: No (already normalized)")

STEP 1: Unicode Normalization

[Tweet 1]
  INPUT :      .Assalam Alikum 🦋😊 اے ایمان والو میرے دشمنوں اور اپنے دشمنوں کافروں کو دوست نہ بناؤ کہ ان کے پاس دوستی کے پ…
  OUTPUT:      .Assalam Alikum 🦋😊 اے ایمان والو میرے دشمنوں اور اپنے دشمنوں کافروں کو دوست نہ بناؤ کہ ان کے پاس دوستی کے پ…
  >> Changed: No (already normalized)

[Tweet 2]
  INPUT :  - '' 🍀 🌾 🌴 '' - ﺍﮬﺪِﻧَﺎﻟﺼِّﺮَﺍﻁَ ﺍَﻟﻤُﺴﺘَﻘِﯿﻢَ '' ﺍﮮ ﻣﯿﺮﮮ ﺭﺏ، ﻣﺠﮭﮯ ﺳﯿﺪﮬﮯ ﺭﺍﺳﺘﮯ ﭘﺮ ﭼﻼ❤️السلام و عليكم.. صباح الخير✨ جمعہ مبارک..💝…
  OUTPUT:  - '' 🍀 🌾 🌴 '' - ﺍﮬﺪِﻧَﺎﻟﺼِّﺮَﺍﻁَ ﺍَﻟﻤُﺴﺘَﻘِﯿﻢَ '' ﺍﮮ ﻣﯿﺮﮮ ﺭﺏ، ﻣﺠﮭﮯ ﺳﯿﺪﮬﮯ ﺭﺍﺳﺘﮯ ﭘﺮ ﭼﻼ❤️السلام و عليكم.. صباح الخير✨ جمعہ مبارک..💝…
  >> Changed: No (already normalized)

[Tweet 3]
  INPUT :    💕 میں نہیں کرتی اُس کا ذکر کسی تیسرے کے ساتھ اُس کے بارے میں بات صرف خُدا سے ہوتی ہے   .💕 نـــور
  OUTPUT:    💕 میں نہیں کرتی اُس کا ذکر کسی تیسرے کے ساتھ اُس کے بارے میں بات صرف خُدا سے ہوتی ہے   .💕 نـــور
  >> Changed: No (already normalized)

[Tweet 4]
  INPUT :   .🌿.. ﴿کے میشود تمامِ جـهان سـوےِ کربـلا با کاروانِ

### Step 2: URL Removal

**Why:** Tweets often contain HTTP/HTTPS links that carry no sentiment-relevant linguistic content. URLs are unique per tweet and would pollute vocabulary with noise.

**What it does:** Removes all URLs matching `http://`, `https://`, or `www.` patterns.

In [6]:
def remove_urls(text):
    """Remove all HTTP/HTTPS URLs and www-links."""
    return re.sub(r"https?://\S+|www\.\S+", "", text)


# Demonstrate on sample
print("STEP 2: URL Removal")
print("=" * 70)
# Use a manual example with URL since dataset tweets may not have URLs
test_text = "یہ خبر پڑھیں https://t.co/abc123 بہت اہم ہے"
print(f"  INPUT : {test_text}")
print(f"  OUTPUT: {remove_urls(test_text)}")

# Also show on actual dataset tweets
for i, row in sample_df.iterrows():
    original = row['Text'][:150]
    cleaned = remove_urls(original)
    if original != cleaned:
        print(f"\n[Tweet {i+1}]")
        print(f"  INPUT : {original}")
        print(f"  OUTPUT: {cleaned}")

STEP 2: URL Removal
  INPUT : یہ خبر پڑھیں https://t.co/abc123 بہت اہم ہے
  OUTPUT: یہ خبر پڑھیں  بہت اہم ہے


### Step 3: Mention (@username) Removal

**Why:** Twitter @mentions are user handles, not linguistic content. Keeping them would cause the model to memorize spurious correlations between specific users and sentiments.

**What it does:** Removes all `@username` patterns.

In [7]:
def remove_mentions(text):
    """Remove Twitter @username mentions."""
    return re.sub(r"@\w+", "", text)


# Demonstrate
print("STEP 3: Mention Removal")
print("=" * 70)
test_text = "@AliPK یہ واقعی خوبصورت لمحہ ہے @ZaraPK"
print(f"  INPUT : {test_text}")
print(f"  OUTPUT: {remove_mentions(test_text)}")

STEP 3: Mention Removal
  INPUT : @AliPK یہ واقعی خوبصورت لمحہ ہے @ZaraPK
  OUTPUT:  یہ واقعی خوبصورت لمحہ ہے 


### Step 4: Hashtag Symbol Cleanup

**Why:** Hashtags in Urdu tweets often carry meaningful topical or sentiment-bearing content (e.g., `#خوشی` meaning "happiness"). We remove the `#` symbol but keep the word.

**What it does:** Strips the `#` character while preserving the hashtag word as a regular token.

In [8]:
def clean_hashtags(text):
    """Remove '#' symbol but keep the hashtag word as a regular token."""
    return re.sub(r"#(\S+)", r"\1", text)


# Demonstrate
print("STEP 4: Hashtag Symbol Cleanup")
print("=" * 70)
test_text = "آج کا دن بہت اچھا #خوشی #Pakistan2024"
print(f"  INPUT : {test_text}")
print(f"  OUTPUT: {clean_hashtags(test_text)}")

STEP 4: Hashtag Symbol Cleanup
  INPUT : آج کا دن بہت اچھا #خوشی #Pakistan2024
  OUTPUT: آج کا دن بہت اچھا خوشی Pakistan2024


### Step 5: Emoji Removal (Label-Leakage Prevention) ⚠️ CRITICAL

**Why:** This is the **most important** preprocessing step for this dataset. The SentiUrdu-1M labels were assigned based on emoticons — a positive emoji caused a tweet to receive a positive label. If we retain emojis as input features, the model will trivially learn to predict sentiment from the same emoticons used to generate the labels, producing **artificially high accuracy** that does not reflect actual language understanding.

**What it does:** Removes all emoji characters from the tweet text to prevent label-signal leakage.

In [9]:
def remove_emojis(text):
    """
    Remove all emoji characters from tweet text.
    
    RATIONALE: SentiUrdu-1M labels were assigned using emoticons as the
    weak supervision signal. Keeping emojis as model input would leak the
    labeling heuristic into the feature space, causing the model to learn
    from the label source rather than Urdu linguistic content.
    """
    return emoji.replace_emoji(text, replace="")


# Demonstrate on actual dataset tweets (which contain emojis)
print("STEP 5: Emoji Removal (Label-Leakage Prevention)")
print("=" * 70)
for i, row in sample_df.iterrows():
    original = row['Text'][:150]
    cleaned = remove_emojis(original)
    if original != cleaned:
        print(f"\n[Tweet {i+1}]")
        print(f"  INPUT : {original}")
        print(f"  OUTPUT: {cleaned}")

STEP 5: Emoji Removal (Label-Leakage Prevention)

[Tweet 1]
  INPUT :      .Assalam Alikum 🦋😊 اے ایمان والو میرے دشمنوں اور اپنے دشمنوں کافروں کو دوست نہ بناؤ کہ ان کے پاس دوستی کے پ…
  OUTPUT:      .Assalam Alikum  اے ایمان والو میرے دشمنوں اور اپنے دشمنوں کافروں کو دوست نہ بناؤ کہ ان کے پاس دوستی کے پ…

[Tweet 2]
  INPUT :  - '' 🍀 🌾 🌴 '' - ﺍﮬﺪِﻧَﺎﻟﺼِّﺮَﺍﻁَ ﺍَﻟﻤُﺴﺘَﻘِﯿﻢَ '' ﺍﮮ ﻣﯿﺮﮮ ﺭﺏ، ﻣﺠﮭﮯ ﺳﯿﺪﮬﮯ ﺭﺍﺳﺘﮯ ﭘﺮ ﭼﻼ❤️السلام و عليكم.. صباح الخير✨ جمعہ مبارک..💝…
  OUTPUT:  - ''    '' - ﺍﮬﺪِﻧَﺎﻟﺼِّﺮَﺍﻁَ ﺍَﻟﻤُﺴﺘَﻘِﯿﻢَ '' ﺍﮮ ﻣﯿﺮﮮ ﺭﺏ، ﻣﺠﮭﮯ ﺳﯿﺪﮬﮯ ﺭﺍﺳﺘﮯ ﭘﺮ ﭼﻼالسلام و عليكم.. صباح الخير جمعہ مبارک..…

[Tweet 3]
  INPUT :    💕 میں نہیں کرتی اُس کا ذکر کسی تیسرے کے ساتھ اُس کے بارے میں بات صرف خُدا سے ہوتی ہے   .💕 نـــور
  OUTPUT:     میں نہیں کرتی اُس کا ذکر کسی تیسرے کے ساتھ اُس کے بارے میں بات صرف خُدا سے ہوتی ہے   . نـــور

[Tweet 4]
  INPUT :   .🌿.. ﴿کے میشود تمامِ جـهان سـوےِ کربـلا با کاروانِ مهـدےِ موعد عج سفـر کنند﴾ حب_الحسین_یجمعنا
  OUTPUT:   ... ﴿کے میشود تمامِ جـهان سـوےِ کربـلا با کاروانِ 

### Step 6: Number Removal

**Why:** Both Western digits (0-9) and Eastern Arabic-Indic digits (٠-٩) are removed, as numbers rarely carry sentiment polarity and add unnecessary vocabulary diversity.

**What it does:** Removes all numeric digits (Western and Eastern) from the text.

In [10]:
def remove_numbers(text):
    """Remove Western digits (0-9) and Eastern Arabic-Indic digits."""
    return re.sub(r"[\d٠-٩۰-۹]", "", text)


# Demonstrate
print("STEP 6: Number Removal")
print("=" * 70)
test_text = "حکومت کی 5 سالوں میں 2024 بہتری نہیں آئی ۲۰۲۴"
print(f"  INPUT : {test_text}")
print(f"  OUTPUT: {remove_numbers(test_text)}")

STEP 6: Number Removal
  INPUT : حکومت کی 5 سالوں میں 2024 بہتری نہیں آئی ۲۰۲۴
  OUTPUT: حکومت کی  سالوں میں  بہتری نہیں آئی 


### Step 7: Punctuation Removal

**Why:** ASCII punctuation and repetitive Urdu punctuation marks (such as `!!!` or `؟؟؟`) are noise rather than morphological features. We remove them while preserving Urdu script letters and spaces.

**What it does:** Removes ASCII punctuation and Urdu/Arabic punctuation marks.

In [11]:
def remove_punctuation(text):
    """
    Remove ASCII punctuation and repetitive Urdu/Arabic punctuation marks.
    Urdu script letters (Unicode block U+0600-U+06FF) are preserved.
    """
    # Remove standard ASCII punctuation
    text = re.sub(r'[!"#$%&\'()*+,\-./:;<=>?@\[\\\]^_`{|}~]', "", text)
    # Remove Urdu/Arabic punctuation marks
    text = re.sub(r"[\u06D4\u060C\u061F\u061B\u066A\u066B\u066C]+", " ", text)
    return text


# Demonstrate
print("STEP 7: Punctuation Removal")
print("=" * 70)
test_text = "عجیب بات ہے۔ کیا آپ کو معلوم تھا؟ یہ!!! سچ ہے..."
print(f"  INPUT : {test_text}")
print(f"  OUTPUT: {remove_punctuation(test_text)}")

STEP 7: Punctuation Removal
  INPUT : عجیب بات ہے۔ کیا آپ کو معلوم تھا؟ یہ!!! سچ ہے...
  OUTPUT: عجیب بات ہے  کیا آپ کو معلوم تھا  یہ سچ ہے


### Step 8: Whitespace Normalization

**Why:** After all removal steps, multiple consecutive spaces, tabs, or newlines may remain. We collapse them into a single space for clean, consistently formatted text ready for tokenization.

**What it does:** Replaces all sequences of whitespace with a single space and strips leading/trailing spaces.

In [12]:
def normalize_whitespace(text):
    """Collapse multiple spaces/tabs/newlines into a single space and strip."""
    return re.sub(r"\s+", " ", text).strip()


# Demonstrate
print("STEP 8: Whitespace Normalization")
print("=" * 70)
test_text = "  آج   کا    دن   بہت   اچھا   تھا  "
print(f"  INPUT : '{test_text}'")
print(f"  OUTPUT: '{normalize_whitespace(test_text)}'")

STEP 8: Whitespace Normalization
  INPUT : '  آج   کا    دن   بہت   اچھا   تھا  '
  OUTPUT: 'آج کا دن بہت اچھا تھا'


---

## Complete Preprocessing Pipeline

All 8 steps combined into a single pipeline function that processes a tweet through each stage in order.

In [13]:
def preprocess_tweet(text):
    """
    Apply the full 8-step preprocessing pipeline to a single Urdu tweet.
    
    Pipeline Order:
        1. Unicode normalization (NFC)
        2. URL removal
        3. @mention removal
        4. Hashtag symbol cleanup
        5. Emoji removal (label-leakage prevention)
        6. Number removal
        7. Punctuation removal
        8. Whitespace normalization
    
    Args:
        text (str): Raw Urdu tweet string.
    
    Returns:
        str: Cleaned tweet ready for tokenization or vectorization.
    """
    text = normalize_unicode(text)
    text = remove_urls(text)
    text = remove_mentions(text)
    text = clean_hashtags(text)
    text = remove_emojis(text)
    text = remove_numbers(text)
    text = remove_punctuation(text)
    text = normalize_whitespace(text)
    return text


print("Pipeline function defined successfully.")

Pipeline function defined successfully.


---

## Demo: Full Pipeline on Sample Tweets

Applying the complete pipeline on 3 manually crafted example tweets to clearly show the input → output transformation.

In [14]:
# 3 manually crafted sample tweets that demonstrate all preprocessing steps
sample_tweets = [
    # Tweet 1: Positive emoji, URL, mention, hashtag
    "آج کا دن بہت اچھا تھا 😊 @Ali_PK یہ واقعی خوبصورت لمحہ ہے https://t.co/xyz123 #خوشی",

    # Tweet 2: Negative emojis, number, ASCII punctuation, URL
    "حکومت کی پالیسیاں بالکل غلط ہیں 😡😡 کوئی بہتری نہیں آئی 5 سالوں میں!!! https://bit.ly/abc",

    # Tweet 3: Neutral emoji, mention, code-mixing, Urdu punctuation
    "@ZaraPK یہ news سن کر بہت حیرت ہوئی 😐 #Pakistan2024 واقعی عجیب بات ہے۔ کیا آپ کو معلوم تھا؟",
]

print("=" * 65)
print("  Urdu Tweet Preprocessing Demo — Full Pipeline")
print("=" * 65)
for i, tweet in enumerate(sample_tweets, 1):
    cleaned = preprocess_tweet(tweet)
    print(f"\n[Tweet {i}]")
    print(f"  RAW    : {tweet}")
    print(f"  CLEANED: {cleaned}")
print("\n" + "=" * 65)

  Urdu Tweet Preprocessing Demo — Full Pipeline

[Tweet 1]
  RAW    : آج کا دن بہت اچھا تھا 😊 @Ali_PK یہ واقعی خوبصورت لمحہ ہے https://t.co/xyz123 #خوشی
  CLEANED: آج کا دن بہت اچھا تھا یہ واقعی خوبصورت لمحہ ہے خوشی

[Tweet 2]
  RAW    : حکومت کی پالیسیاں بالکل غلط ہیں 😡😡 کوئی بہتری نہیں آئی 5 سالوں میں!!! https://bit.ly/abc
  CLEANED: حکومت کی پالیسیاں بالکل غلط ہیں کوئی بہتری نہیں آئی سالوں میں

[Tweet 3]
  RAW    : @ZaraPK یہ news سن کر بہت حیرت ہوئی 😐 #Pakistan2024 واقعی عجیب بات ہے۔ کیا آپ کو معلوم تھا؟
  CLEANED: یہ news سن کر بہت حیرت ہوئی Pakistan واقعی عجیب بات ہے کیا آپ کو معلوم تھا



### Expected Output

```
=================================================================
  Urdu Tweet Preprocessing Demo — Full Pipeline
=================================================================

[Tweet 1]
  RAW    : آج کا دن بہت اچھا تھا 😊 @Ali_PK یہ واقعی خوبصورت لمحہ ہے https://t.co/xyz123 #خوشی
  CLEANED: آج کا دن بہت اچھا تھا یہ واقعی خوبصورت لمحہ ہے خوشی

[Tweet 2]
  RAW    : حکومت کی پالیسیاں بالکل غلط ہیں 😡😡 کوئی بہتری نہیں آئی 5 سالوں میں!!! https://bit.ly/abc
  CLEANED: حکومت کی پالیسیاں بالکل غلط ہیں کوئی بہتری نہیں آئی سالوں میں

[Tweet 3]
  RAW    : @ZaraPK یہ news سن کر بہت حیرت ہوئی 😐 #Pakistan2024 واقعی عجیب بات ہے۔ کیا آپ کو معلوم تھا؟
  CLEANED: یہ news سن کر بہت حیرت ہوئی Pakistan واقعی عجیب بات ہے کیا آپ کو معلوم تھا

=================================================================
```

---

## Apply Pipeline to Actual Dataset

Now we apply the full pipeline to the actual Urdu Tweets Dataset and show before/after comparison.

In [15]:
# Apply preprocessing to a sample of 1000 tweets from the actual dataset
sample_size = 1000
df_sample = df[['Text', 'Emotions', 'Category']].dropna(subset=['Text']).head(sample_size).copy()

# Apply the pipeline
df_sample['Cleaned_Text'] = df_sample['Text'].apply(preprocess_tweet)

print(f"Processed {len(df_sample)} tweets from the dataset.")
print(f"\n{'='*70}")
print("BEFORE vs AFTER Preprocessing (first 5 tweets)")
print(f"{'='*70}")

for i in range(5):
    print(f"\n--- Tweet {i+1} ---")
    print(f"  BEFORE: {df_sample.iloc[i]['Text'][:150]}")
    print(f"  AFTER : {df_sample.iloc[i]['Cleaned_Text'][:150]}")

Processed 1000 tweets from the dataset.

BEFORE vs AFTER Preprocessing (first 5 tweets)

--- Tweet 1 ---
  BEFORE:      .Assalam Alikum 🦋😊 اے ایمان والو میرے دشمنوں اور اپنے دشمنوں کافروں کو دوست نہ بناؤ کہ ان کے پاس دوستی کے پ…
  AFTER : Assalam Alikum اے ایمان والو میرے دشمنوں اور اپنے دشمنوں کافروں کو دوست نہ بناؤ کہ ان کے پاس دوستی کے پ…

--- Tweet 2 ---
  BEFORE:  - '' 🍀 🌾 🌴 '' - ﺍﮬﺪِﻧَﺎﻟﺼِّﺮَﺍﻁَ ﺍَﻟﻤُﺴﺘَﻘِﯿﻢَ '' ﺍﮮ ﻣﯿﺮﮮ ﺭﺏ، ﻣﺠﮭﮯ ﺳﯿﺪﮬﮯ ﺭﺍﺳﺘﮯ ﭘﺮ ﭼﻼ❤️السلام و عليكم.. صباح الخير✨ جمعہ مبارک..💝…
  AFTER : ﺍﮬﺪِﻧَﺎﻟﺼِّﺮَﺍﻁَ ﺍَﻟﻤُﺴﺘَﻘِﯿﻢَ ﺍﮮ ﻣﯿﺮﮮ ﺭﺏ ﻣﺠﮭﮯ ﺳﯿﺪﮬﮯ ﺭﺍﺳﺘﮯ ﭘﺮ ﭼﻼالسلام و عليكم صباح الخير جمعہ مبارک…

--- Tweet 3 ---
  BEFORE:    💕 میں نہیں کرتی اُس کا ذکر کسی تیسرے کے ساتھ اُس کے بارے میں بات صرف خُدا سے ہوتی ہے   .💕 نـــور
  AFTER : میں نہیں کرتی اُس کا ذکر کسی تیسرے کے ساتھ اُس کے بارے میں بات صرف خُدا سے ہوتی ہے نـــور

--- Tweet 4 ---
  BEFORE:   .🌿.. ﴿کے میشود تمامِ جـهان سـوےِ کربـلا با کاروانِ مهـدےِ موعد عج سفـر کنند﴾ حب_الحسین_یجمعنا
  AFTER : ﴿کے میشود تمامِ جـهان سـوے

## Summary Statistics After Preprocessing

In [16]:
# Calculate statistics
df_sample['original_len'] = df_sample['Text'].apply(len)
df_sample['cleaned_len'] = df_sample['Cleaned_Text'].apply(len)
df_sample['reduction'] = df_sample['original_len'] - df_sample['cleaned_len']

# Count empty tweets after cleaning
empty_count = (df_sample['Cleaned_Text'].str.strip() == '').sum()

print("=" * 50)
print("  Preprocessing Statistics")
print("=" * 50)
print(f"  Total tweets processed : {len(df_sample)}")
print(f"  Avg original length    : {df_sample['original_len'].mean():.1f} chars")
print(f"  Avg cleaned length     : {df_sample['cleaned_len'].mean():.1f} chars")
print(f"  Avg reduction          : {df_sample['reduction'].mean():.1f} chars")
print(f"  Empty after cleaning   : {empty_count}")
print("=" * 50)

  Preprocessing Statistics
  Total tweets processed : 1000
  Avg original length    : 94.4 chars
  Avg cleaned length     : 83.9 chars
  Avg reduction          : 10.6 chars
  Empty after cleaning   : 1


## Final Cleaned Dataset Preview

In [17]:
# Show final cleaned dataset
print("Final cleaned dataset sample:")
df_sample[['Text', 'Cleaned_Text', 'Emotions', 'Category']].head(10)

Final cleaned dataset sample:


,Text,Cleaned_Text,Emotions,Category
0,.Assalam Alikum 🦋😊 اے ایمان والو میرے دشم...,Assalam Alikum اے ایمان والو میرے دشمنوں اور ا...,"['SMILING FACE WITH SMILING EYES , 0.644']",Joy
1,- '' 🍀 🌾 🌴 '' - ﺍﮬﺪِﻧَﺎﻟﺼِّﺮَﺍﻁَ ﺍَﻟﻤُﺴﺘَﻘِﯿﻢ...,ﺍﮬﺪِﻧَﺎﻟﺼِّﺮَﺍﻁَ ﺍَﻟﻤُﺴﺘَﻘِﯿﻢَ ﺍﮮ ﻣﯿﺮﮮ ﺭﺏ ﻣﺠﮭﮯ...,"['HEAVY BLACK HEART , 0.746', 'SPARKLES , 0.35...",NaN
2,💕 میں نہیں کرتی اُس کا ذکر کسی تیسرے کے سات...,میں نہیں کرتی اُس کا ذکر کسی تیسرے کے ساتھ اُس...,"['TWO HEARTS , 0.632']",NaN
3,.🌿.. ﴿کے میشود تمامِ جـهان سـوےِ کربـلا با ک...,﴿کے میشود تمامِ جـهان سـوےِ کربـلا با کاروانِ ...,"['HERB , 0.384']",NaN
4,.🌿.. ﴿کے میشود تمامِ جـهان سـوےِ کربـلا با ک...,﴿کے میشود تمامِ جـهان سـوےِ کربـلا با کاروانِ ...,"['HERB , 0.384']",NaN
5,.🌿.. ﴿کے میشود تمامِ جـهان سـوےِ کربـلا با ک...,﴿کے میشود تمامِ جـهان سـوےِ کربـلا با کاروانِ ...,"['HERB , 0.384']",NaN
6,ْ .💌 ْ ﮼اللّہُم،صلِّ،وسلمِّ،على،نبيِّنا،مُحمَ...,ْ ْ ﮼اللّہُم صلِّ وسلمِّ على نبيِّنا مُحمَّد ♡ ٰ,"['WHITE HEART SUIT , 0.669', 'LOVE LETTER , 0.5']",NaN
7,.💕کوئی گلہ کوئی شکوہ نہ رہے آپ سے یہ آرزو ہے...,کوئی گلہ کوئی شکوہ نہ رہے آپ سے یہ آرزو ہے اک ...,"['TWO HEARTS , 0.632']",NaN
8,-- /👑 واحد اللہ کا در ہے جہاں بغیر بلاوے کے ب...,واحد اللہ کا در ہے جہاں بغیر بلاوے کے بھی چلے ...,"['CROWN , 0.694']",NaN
9,'' __________🌼✨🍁 کبھی جو میرا ضبط ٹُوٹے میں ت...,کبھی جو میرا ضبط ٹُوٹے میں تُجھ سے رابطہ کر لوں,"['SPARKLES , 0.351', 'BLOSSOM , 0.779', 'MAPLE...",NaN


---

## Steps NOT Applied and Why

| Step | Reason for Exclusion |
|------|---------------------|
| **Lowercasing** | Urdu script has no upper/lower case distinction. Not applicable. |
| **Stemming** | Urdu morphology is complex and no reliable Urdu stemmer exists. Incorrect stemming would damage meaning. |
| **Lemmatization** | Same as stemming — no mature Urdu lemmatizer available; premature application would harm model quality. |
| **Stopword Removal** | Urdu stopword lists are not standardized. Removing words prematurely can strip contextual meaning from short tweets. Will be reconsidered during feature engineering phase. |